SEPARATING DATASET POSITIVE AND NEGATIVE GON PATIENTS

In [30]:
import os
import shutil
import pandas as pd

df = pd.read_csv("Labels.csv")
df = df[df["Quality Score"] >= 4]


In [ ]:
image_folder = "Images"
output_folder = "Classified_Images"

for label in ["GON+", "GON-"]:
    os.makedirs(os.path.join(output_folder, label), exist_ok=True)

for _, row in df.iterrows():
    img_name = row["Image Name"]
    label = row["Label"]
    
    src = os.path.join(image_folder, img_name)
    dst = os.path.join(output_folder, label, img_name)
    
    if os.path.exists(src):
        shutil.copy(src.replace('\\','/'), dst.replace('\\','/'))

print("Done classifying images (filtered + separated)!")

Done classifying images (filtered + separated)!


RESIZING IMAGES DATASET

In [10]:
import cv2
import numpy as np
import os

input_root = 'Classified_Images'
output_root = 'PROCESSED_224'
subfolders = ['GON-', 'GON+']
target_size = (224, 224)

# CLAHE object (used for enhancement)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)) 

def auto_crop_and_process(img):
    # Convert to grayscale for cropping detection
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Threshold to separate object from background
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    
    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Crop based on largest contour
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest_contour)
        
        size = max(w, h)
        cx, cy = x + w//2, y + h//2
        half_size = size // 2
        
        start_x = max(0, cx - half_size)
        end_x = min(img.shape[1], cx + half_size)
        start_y = max(0, cy - half_size)
        end_y = min(img.shape[0], cy + half_size)
        
        img_cropped = img[start_y:end_y, start_x:end_x]
    else:
        img_cropped = img

    # ENHANCEMENT (CLAHE on grayscale)
    gray_cropped = cv2.cvtColor(img_cropped, cv2.COLOR_BGR2GRAY)
    # Apply CLAHE to improve contrast
    enhanced_gray = clahe.apply(gray_cropped)
    # END OF ENHANCEMENT
    # Resize to target size
    final_img = cv2.resize(enhanced_gray, target_size, interpolation=cv2.INTER_AREA)
    
    return final_img

if not os.path.exists(output_root):
    os.makedirs(output_root)

for sub in subfolders:
    input_path = os.path.join(input_root, sub)
    output_path = os.path.join(output_root, sub)
    
    if not os.path.exists(output_path):
        os.makedirs(output_path)
    
    print(f"Processing folder: {sub}...")
    
    files = [f for f in os.listdir(input_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    for filename in files:
        img_path = os.path.join(input_path, filename)
        img = cv2.imread(img_path)
        
        if img is not None:
            final_img = auto_crop_and_process(img)
            
            save_name = os.path.splitext(filename)[0] + '.jpg'
            cv2.imwrite(os.path.join(output_path, save_name), final_img)

print("DONEE")

Processing folder: GON-...
Processing folder: GON+...
DONEE


SPLITTING DATA 72% TRAINING AND 18% MODEL VALIDATION 10% TESTING, SPLITTING FOR 224 SIZE IMAGE

In [31]:
import pandas as pd

image_folder = './PROCESSED_224/'

for file_path in df['Image Name']:
  gon_value = df.loc[df['Image Name']==file_path, 'Label'].values[0]
  file_path_result = image_folder+gon_value+'/'+file_path
  df['Image Name'].replace(file_path, file_path_result, inplace=True)

from sklearn.model_selection import train_test_split

#separating the labels with the image
image_file = df['Image Name']
labels = df['Label']

#converting the labels with binaries value
for value in labels:
    if value == "GON-":
        labels.replace(value, 0, inplace=True)
    if value == "GON+":
        labels.replace(value, 1, inplace=True)

#splitting the image and the labels for training + validation and testing 90 to 10 percent ratio
train_validation_image, test_image, train_validation_labels, test_labels = train_test_split(image_file, labels, random_state=100, test_size=0.1)

#splitting the image and the labels for training and validation 80 to 20 percent ratio from the traingin_and_validation part
train_image, validation_image, train_labels, validation_labels = train_test_split(train_validation_image, train_validation_labels, random_state=100, test_size=0.2)

train_dataframe = pd.DataFrame({
    'filename': train_image,
    'labels' : train_labels
})

validation_dataframe = pd.DataFrame({
    'filename' : validation_image,
    'labels' : validation_labels
})
test_dataframe = pd.DataFrame({
    'filename': test_image,
    'labels' : test_labels
})

test_dataframe['labels'] = test_dataframe['labels'].astype(str)
train_dataframe['labels'] = train_dataframe['labels'].astype(str)
validation_dataframe['labels'] = validation_dataframe['labels'].astype(str)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17404\509262295.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Image Name'].replace(file_path, file_path_result, inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_17404\509262295.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  labels.re

MODEL TRAINING AND VALIDATION 

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import numpy as np
import matplotlib.pyplot as plt

image_size = 224

base_model = tf.keras.applications.EfficientNetV2S(weights='imagenet', input_shape=(image_size, image_size, 3), include_top=False)

base_model.trainable = True

model = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'), #i use the 1024 for the first model and use the 512 for the second model
    layers.Dropout(0.7),
    layers.Dense(1, activation='sigmoid')
])

from tensorflow.keras.optimizers import Adam

adam_opt = Adam(learning_rate=0.0001)

model.compile(optimizer=adam_opt, loss='binary_crossentropy', metrics=['accuracy'])

train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0,
    height_shift_range=0,
    zoom_range=0,
    horizontal_flip=True,
    fill_mode='constant',
    cval=0
)

validation_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_dataframe,
    x_col="filename",
    y_col="labels",
    target_size = (image_size, image_size),
    class_mode = 'binary',
    color_mode = 'rgb',
    shuffle=True
)

validation_generator = validation_datagen.flow_from_dataframe(
    dataframe= validation_dataframe,
    x_col="filename",
    y_col="labels",
    class_mode = 'binary',
    color_mode = 'rgb',
    target_size = (image_size, image_size)
)

EPOCHS=50
best_model_file = f"../PROCESSED_224/best_model/best_model_{image_size}_1.h5"

from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

callbacks = [
    ModelCheckpoint(best_model_file, verbose=1, save_best_only=True, monitor="val_loss"),
    ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.1, verbose=1, min_lr=1e-6),
    EarlyStopping(monitor='val_loss', patience=7, verbose=1)
]

result = model.fit(
    train_generator, epochs=EPOCHS, validation_data=validation_generator, callbacks=callbacks
)

best_val_ac_epoch = np.argmax(result.history['val_accuracy'])
best_val_acc = result.history['val_accuracy'][best_val_ac_epoch]

print("Best validation accuracy : " + str(best_val_acc))

plt.plot(result.history['accuracy'], label='train_acc')
plt.plot(result.history['val_accuracy'], label='val_acc')
plt.legend()
plt.show()


plt.plot(result.history['loss'], label='train_loss')
plt.plot(result.history['val_loss'], label='val_loss')
plt.legend()
plt.show()


MODEL TESTING, TESTING THE MODEL WITH 10% OF ALL DATA

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Prepare the Test Generator (Must be SHUFFLE=FALSE)
test_datagen = ImageDataGenerator()
image_size = 224
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_dataframe, # Using your 10% test split
    x_col="filename",
    y_col="labels",
    target_size=(image_size, image_size),
    class_mode='binary',
    color_mode='rgb',
    batch_size=1, # One at a time for precise evaluation
    shuffle=False  # CRITICAL: Keep images in order
)

# 2. Load your best saved model
print("Loading the best model...")
model = load_model("/content/best_model_224_2.h5")

# 3. Get numerical predictions
# predict() gives values between 0.0 and 1.0
Y_pred = model.predict(test_generator)
# Convert probabilities to binary 0 or 1
y_pred = (Y_pred > 0.2).astype(int).flatten()

# 4. Get the true labels from the generator
y_true = test_generator.classes

# 5. Generate and Plot the Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['GON-', 'GON+'], yticklabels=['GON-', 'GON+'])
plt.title('Confusion Matrix: Final Test Set')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# 6. Print the detailed Classification Report
# This shows your Recall (Sensitivity) and Precision
print("\n--- Detailed Classification Report ---")
print(classification_report(y_true, y_pred, target_names=['GON-', 'GON+']))
